In [9]:
from ingest import load_sec_data
docs = load_sec_data("Shopify")

In [10]:
docs[10]

{'id': '8ee9e1ba5a',
 'company': 'Shopify',
 'section': 'Item 1 - Business',
 'text': 'Our Merchants\nWe believe we can help merchants of all verticals and sizes, from aspirational entrepreneurs to companies with large-scale, direct-to-consumer or business to business ("B2B") operations, or both, realize their potential at all stages of their business life cycle. Our merchants represent a wide array of retail verticals and business sizes and no single merchant has ever represented more than five percent of our total revenues in a single reporting period. As of December 31, 2025, we had millions of merchants from more than 175 countries using our platform, geographically dispersed as follows: 44% in the United States, 31% in Europe, the Middle East and Africa, 16% in Asia Pacific, Australia and China, 5% in Canada and 5% in Latin America.'}

In [11]:
documents_llm = []

for doc in docs:
    if doc["company"] == "Shopify":
        documents_llm.append(doc)

len(documents_llm)

532

In [12]:
docs = documents_llm

In [13]:
doc = docs[100]
print(doc["id"])
print(doc["section"])
print(doc["text"])

dbf7f11c7c
Item 1 - Business
We have in the past made, and in the future may make, acquisitions, divestitures and investments, which could divert management’s attention, result in operating difficulties and dilution to our shareholders and otherwise disrupt our operations and adversely affect our business, operating results or financial position.
From time to time, we evaluate potential acquisitions, divestitures and strategic investment opportunities to support our business initiatives. Any transactions that we enter into could be material to our financial condition and results of operations. Acquisitions, divestitures and investments involve a number of risks, such as:
• diversion of management time and focus from operating our business;
• use of resources that are needed in other areas of our business;
• in the case of an acquisition:
◦ implementation or remediation of controls, procedures and policies of the acquired company;
◦ difficulty integrating the operations of the acquired 

In [14]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [15]:
data_gen_instructions = """
You emulate a retail investor or financial analyst researching a public company.
Formulate 3 to 5 questions this person might ask based on an SEC 10-K filing excerpt.

The excerpt must contain the exact facts, figures, or risks to answer the questions.
Each question must be self-contained and explicit, mentioning the company name and fiscal year/period where relevant.
Use as few words as possible verbatim from the excerpt to avoid direct lexical overlap.
The output should resemble how investors or analysts discuss stocks on forums like Reddit, Twitter/X, or Seeking Alpha—practical, focused on business metrics or risks, not overly formal, and neither too short nor too long.
""".strip()

In [16]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [17]:
user_prompt = f"""
Section: {doc['section']}

Filing Excerpt:
{doc['text']}
""".strip()

In [18]:
user_prompt

'Section: Item 1 - Business\n\nFiling Excerpt:\nWe have in the past made, and in the future may make, acquisitions, divestitures and investments, which could divert management’s attention, result in operating difficulties and dilution to our shareholders and otherwise disrupt our operations and adversely affect our business, operating results or financial position.\nFrom time to time, we evaluate potential acquisitions, divestitures and strategic investment opportunities to support our business initiatives. Any transactions that we enter into could be material to our financial condition and results of operations. Acquisitions, divestitures and investments involve a number of risks, such as:\n• diversion of management time and focus from operating our business;\n• use of resources that are needed in other areas of our business;\n• in the case of an acquisition:\n◦ implementation or remediation of controls, procedures and policies of the acquired company;\n◦ difficulty integrating the op

In [19]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [20]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [21]:
response.output_parsed.questions

['For this company, how much could future acquisitions or divestitures in the next fiscal year distract management from the core business, and what’s the actual plan/timeline for those deals?',
 'If the company does a material acquisition, how big could the hit be from integration problems or having to fix the target’s controls and policies, and has management seen this issue before?',
 'Could upcoming investment or M&A activity force the company to pull resources away from higher-priority areas, and which parts of the business would take the biggest hit?',
 'How much dilution should investors expect if the company keeps using acquisitions and strategic investments to grow, especially if any deal is large enough to move the needle on FY results?']

In [22]:
doc

{'id': 'dbf7f11c7c',
 'company': 'Shopify',
 'section': 'Item 1 - Business',
 'text': 'We have in the past made, and in the future may make, acquisitions, divestitures and investments, which could divert management’s attention, result in operating difficulties and dilution to our shareholders and otherwise disrupt our operations and adversely affect our business, operating results or financial position.\nFrom time to time, we evaluate potential acquisitions, divestitures and strategic investment opportunities to support our business initiatives. Any transactions that we enter into could be material to our financial condition and results of operations. Acquisitions, divestitures and investments involve a number of risks, such as:\n• diversion of management time and focus from operating our business;\n• use of resources that are needed in other areas of our business;\n• in the case of an acquisition:\n◦ implementation or remediation of controls, procedures and policies of the acquired co

In [23]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

c:\Users\GIA DAT\fiscal-query\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['For this company in fiscal 2024, how much risk is there that acquisitions or divestitures could pull management away from core operations and hurt execution?', 'If the company does a deal in fiscal 2024, what’s the chance it ends up causing operating problems or forcing extra spending on controls and process fixes at the target?', 'How likely is dilution to existing shareholders from the company’s acquisition or investment plans, and could that meaningfully hit the stock in fiscal 2024?', 'Could any acquisition, divestiture, or strategic investment be big enough to move the company’s 2024 financial condition or results of operations in a noticeable way?']


In [24]:
usage

ResponseUsage(input_tokens=368, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=144, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=512)

In [25]:
from evaluation_utils import calc_price

calc_price(usage)

{'input_cost': 0.000276, 'output_cost': 0.000648, 'total_cost': 0.000924}

In [26]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'For this company in fiscal 2024, how much risk is there that acquisitions or divestitures could pull management away from core operations and hurt execution?',
  'document': 'dbf7f11c7c'},
 {'question': 'If the company does a deal in fiscal 2024, what’s the chance it ends up causing operating problems or forcing extra spending on controls and process fixes at the target?',
  'document': 'dbf7f11c7c'},
 {'question': 'How likely is dilution to existing shareholders from the company’s acquisition or investment plans, and could that meaningfully hit the stock in fiscal 2024?',
  'document': 'dbf7f11c7c'},
 {'question': 'Could any acquisition, divestiture, or strategic investment be big enough to move the company’s 2024 financial condition or results of operations in a noticeable way?',
  'document': 'dbf7f11c7c'}]

In [28]:
import pandas as pd

pd.DataFrame(records)

,question,document
0,"For this company in fiscal 2024, how much risk...",dbf7f11c7c
1,"If the company does a deal in fiscal 2024, wha...",dbf7f11c7c
2,How likely is dilution to existing shareholder...,dbf7f11c7c
3,"Could any acquisition, divestiture, or strateg...",dbf7f11c7c


In [29]:
from evaluation_utils import llm_structured_retry

In [30]:
def generate_ground_truth(doc):
    user_prompt = f"""
                    Section: {doc['section']}

                    Filing Excerpt:
                    {doc['text']}
                    """.strip()

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [31]:
generate_ground_truth(doc)

([{'question': 'For this company’s FY disclosure, how worried should investors be that future acquisitions or divestitures could pull management off the core business and hurt operating performance?',
   'document': 'dbf7f11c7c'},
  {'question': 'If this company ends up buying another business, what’s the risk that integration headaches or fixing the target’s controls/policies could slow down results or create extra costs?',
   'document': 'dbf7f11c7c'},
  {'question': 'How material could these acquisitions, divestitures, or strategic investments be to this company’s financials if it actually closes one of these deals?',
   'document': 'dbf7f11c7c'},
  {'question': 'Could these kinds of transactions use up cash or other resources needed elsewhere in the company, and what would that mean for margins or growth if management gets distracted?',
   'document': 'dbf7f11c7c'}],
 ResponseUsage(input_tokens=369, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), out

In [33]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(docs[100:105]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:11<00:00,  2.32s/it]


In [34]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [36]:
with ThreadPoolExecutor(max_workers=4) as pool:
    results = map_progress(pool, docs, generate_ground_truth)

100%|██████████| 532/532 [04:57<00:00,  1.79it/s]


In [37]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

2293

In [38]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.5720580000000004

In [39]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.5720580000000004

In [40]:
df_ground_truth = pd.DataFrame(ground_truth)

In [41]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [44]:
pd.set_option('display.max_colwidth', None)
df_ground_truth.head(50)

,question,document
0,"For Shopify in this 10-K, are merchants counted as paying subscription shops only, and how might that definition affect anyone trying to model growth for this fiscal year?",fd971d5913
1,Shopify says it reports in U.S. GAAP and uses U.S. dollars for everything in this annual report—does that mean I can compare margins and revenue trends directly without any currency noise for this period?,fd971d5913
2,"In Shopify's 10-K, when they say 'our solutions' includes both products and services for merchants, how should investors think about the mix if they’re trying to judge recurring revenue vs. service revenue this year?",fd971d5913
3,"Since Shopify's filing says references to 'our merchants' mean unique shops paying for a subscription on the platform, is that a cleaner KPI than raw shop count for tracking platform adoption in this fiscal year?",fd971d5913
4,"For this company’s FY10-K, how much of management’s guidance is basically just forward-looking language, and what specific phrasing does the filing say can tip you off?",1b3b7a37cc
5,"In this company’s annual report, are statements about future expectations, plans, and performance all treated as forward-looking under U.S. and Canadian securities rules, or only certain ones?",1b3b7a37cc
6,"What kinds of assumptions does management say sit behind the company’s forward-looking comments in this 10-K, and how much of that is just based on past trends vs. current conditions?",1b3b7a37cc
7,"When the company uses words like 'may,' 'could,' 'expects,' or 'plans' in this filing, does the report say those are automatically forward-looking for the current fiscal year?",1b3b7a37cc
8,"For Shopify in the fiscal year covered by this 10-K, how much risk is tied to actually growing and keeping merchants, especially if competitors make it harder to retain them?",f43d0ae75c
9,"How dependent is Shopify on expanding beyond current features—more channels, more localized tools, better merchant service, and AI—to keep merchant sales rising in this filing period?",f43d0ae75c
